# SmartBearing CWRU — EDA & Preprocessing FINAL

**Dataset:** CWRU Bearing Fault Detection (Kaggle `brjapon/cwru-bearing-datasets`)  
**Tahap:** Data Engineer → Handoff ke ML Engineer 1 (Random Forest) dan ML Engineer 2 (K-Means)  
**Output:** 4 kelas kondisi, 13 fitur model, pembagian 80:20, `StandardScaler` dari training, CSV scaled dan artefak untuk integrasi Flask.

Notebook ini **tidak melatih model ML**. File input harus berisi 9 fitur statistik dan kolom `fault`, seperti `feature_time_48k_2048_load_1.csv` yang digunakan notebook sumber. Kode preprocessing berada pada file `preprocessing.py` terpisah agar tidak ada dua versi kode yang berbeda.

> ⚠️ Data mentah tidak disertakan. Untuk mendapatkan hasil, tempatkan CSV dataset asli pada `data/raw/` lalu jalankan **Run All**. Semua angka dan grafik dihasilkan dari dataset nyata ketika notebook dijalankan, bukan contoh buatan.

## 0. Persiapan environment dan folder

1. Simpan `preprocessing.py` di root project, notebook di `notebooks/`.
2. Buka terminal pada root project: `pip install pandas numpy scikit-learn joblib matplotlib seaborn ipykernel`.
3. Simpan `feature_time_48k_2048_load_1.csv` dari Kaggle ke `data/raw/` (subfolder juga boleh).
4. Jalankan semua sel secara berurutan.

Unduh dataset: https://www.kaggle.com/datasets/brjapon/cwru-bearing-datasets

Notebook bekerja baik ketika current working directory adalah root project **atau** folder `notebooks/`.

In [ ]:
from pathlib import Path
import sys
import json
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Sesuaikan root agar impor preprocessing.py tidak bergantung pada folder notebook.
CWD = Path.cwd().resolve()
if (CWD / "preprocessing.py").exists():
    ROOT = CWD
elif (CWD.parent / "preprocessing.py").exists():
    ROOT = CWD.parent
else:
    raise FileNotFoundError("preprocessing.py tidak ditemukan. Jalankan notebook dari root project atau folder notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import preprocessing
importlib.reload(preprocessing)
from preprocessing import (
    BASE_FEATURES, ENGINEERED_FEATURES, FEATURES, DEFAULT_PATTERN,
    load_raw, clean, add_fault_type, add_features, run_pipeline, transform_input,
)
DATA_DIR = ROOT / "data"
OUT_DIR = DATA_DIR / "processed"
MODEL_DIR = ROOT / "models"
FILE_PATTERN = DEFAULT_PATTERN  # feature_time_48k_2048_load_1.csv
TARGET = "fault_type"            # 4 kelas: Ball, IR, Normal, OR

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Root:", ROOT)
print("Dataset:", FILE_PATTERN)
print("Target:", TARGET)

## 1. Verifikasi dataset CWRU

Gunakan file CSV **fitur statistik**, bukan sinyal MATLAB `.mat`. Untuk perbandingan yang fair, konfigurasi bawaan mengambil file spesifik *48 kHz, window 2048, load 1* sesuai yang diperiksa dalam notebook asal. Jika file yang dimiliki berbeda, perbarui `FILE_PATTERN` lalu dokumentasikan perubahan konfigurasi.

In [ ]:
input_files = sorted(DATA_DIR.rglob(FILE_PATTERN)) if DATA_DIR.exists() else []
if not input_files:
    raise FileNotFoundError(
        f"Tidak ditemukan {FILE_PATTERN} di {DATA_DIR}. "
        "Ekstrak CSV dari Kaggle ke data/raw/ lalu jalankan ulang."
    )
print("CSV ditemukan:")
for path in input_files:
    print(" -", path.relative_to(ROOT))
raw = load_raw(DATA_DIR, file_pattern=FILE_PATTERN)
print("Bentuk data mentah:", raw.shape)
print("Kolom:", raw.columns.tolist())
display(raw.head())

## 2. Exploratory Data Analysis (EDA)

Analisis dilakukan terhadap sumber sebelum ekstraksi fitur tambahan. Metadata (`source_file`, `sampling_khz`, `load_hp`, `source_row`) **tidak dipakai sebagai input** model.

In [ ]:
print("Jumlah baris:", len(raw))
print("Distribusi kelas detail:")
display(raw["fault"].value_counts().rename_axis("fault").to_frame("jumlah"))
print("Deskriptif sembilan fitur dasar:")
display(raw[BASE_FEATURES].describe().T)
print("Missing setiap fitur / label:")
display(raw[BASE_FEATURES + ["fault"]].isna().sum().rename("missing").to_frame())
print("Duplikat fitur + label:", raw.duplicated(subset=BASE_FEATURES + ["fault"]).sum())

### 2.1 Distribusi 4 kategori utama dan kelas detail

Label yang diterima: `Ball_* → Ball`, `IR_* → IR`, `OR_* → OR`, `Normal_* → Normal`. Label tak dikenal akan menimbulkan error, **bukan** otomatis dianggap Normal.

In [ ]:
labeled = add_fault_type(clean(raw))
fig, axes = plt.subplots(1, 2, figsize=(14, 4.7))
sns.countplot(data=labeled, y="fault", order=labeled["fault"].value_counts().index, ax=axes[0], color="steelblue")
axes[0].set_title("Distribusi label detail")
sns.countplot(data=labeled, x="fault_type", order=["Normal", "Ball", "IR", "OR"], ax=axes[1], color="teal")
axes[1].set_title("Distribusi empat kondisi bearing")
plt.tight_layout()
plt.show()
display(labeled["fault_type"].value_counts().to_frame("jumlah"))

### 2.2 Distribusi fitur, korelasi, dan anomali

Outlier **tidak dihapus otomatis** karena sinyal ekstrem dapat berkaitan dengan gejala kerusakan. Lakukan interpretasi sebelum mengambil keputusan pembersihan lebih jauh.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 10))
for ax, name in zip(axes.flat, BASE_FEATURES):
    sns.histplot(pd.to_numeric(labeled[name], errors="coerce"), kde=True, ax=ax, color="slateblue")
    ax.set_title(name)
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
sns.heatmap(labeled[BASE_FEATURES].corr(), annot=True, fmt=".2f", center=0, cmap="coolwarm")
plt.title("Korelasi sembilan fitur statistik CWRU")
plt.tight_layout()
plt.show()

iqr_rows = []
for name in BASE_FEATURES:
    q1, q3 = labeled[name].quantile([0.25, 0.75])
    width = q3-q1
    count = int(((labeled[name] < q1-1.5*width) | (labeled[name] > q3+1.5*width)).sum())
    iqr_rows.append({"fitur": name, "outlier_IQR": count, "persentase": round(100*count/len(labeled), 2)})
display(pd.DataFrame(iqr_rows).sort_values("persentase", ascending=False))

## 3. Cleaning, label 4 kelas, dan feature engineering

9 fitur statistik asli: `max`, `min`, `mean`, `sd`, `rms`, `skewness`, `kurtosis`, `crest`, `form`.

4 fitur turunan (sama dengan script Data Engineer semula):

| Fitur | Rumus | Keterangan |
|---|---|---|
| peak_to_peak | max − min | Rentang amplitudo |
| abs_peak | max(abs(max), abs(min)) | Puncak mutlak |
| energy | rms² | **Mean-square** amplitudo, bukan total energi seluruh window |
| sd_rms_ratio | sd / (rms + 1e-12) | Rasio variasi terhadap RMS |

Total **13 fitur** untuk kedua modul ML.

In [ ]:
cleaned = add_features(labeled)
print("Awal:", len(raw), "baris  |  Setelah cleaning:", len(cleaned), "baris")
print("Fitur model:", FEATURES)
display(cleaned[FEATURES + ["fault", "fault_type"]].head())

## 4. Split 80:20, scaling pada data training, dan simpan seluruh artefak

**Peringatan evaluasi:** CSV ini berisi fitur per window. `source_file` dalam pilihan bawaan hanya menunjukkan satu CSV dan **bukan** identitas rekaman getaran asal. Karena tidak ada `recording_id` yang diverifikasi, split yang digunakan adalah stratified random **antar baris/window**, bukan holdout independen antar rekaman. Jangan mengklaim skor model membuktikan generalisasi antar rekaman. Jika kelompok memiliki identitas rekaman asli yang terpercaya, gunakan group-aware split sebelum melaporkan evaluasi final.

In [ ]:
result = run_pipeline(
    data_dir=DATA_DIR,
    out_dir=OUT_DIR,
    model_dir=MODEL_DIR,
    target=TARGET,
    test_size=0.20,
    file_pattern=FILE_PATTERN,
    save=True,
)
print("Train:", result["X_train"].shape, "| Test:", result["X_test"].shape)
print("Label encoder:", list(result["label_encoder"].classes_))
print("\nPERINGATAN:", result["split_warning"])
summary = json.loads((OUT_DIR / "preprocessing_summary.json").read_text(encoding="utf-8"))
display(pd.DataFrame([summary]).T.rename(columns={0:"hasil"}))

### 4.1 Memeriksa kelas train dan test (4 kelas)

`label` dalam CSV train/test adalah integer sesuai `LabelEncoder`, dan **tidak termasuk fitur** saat training.

In [ ]:
encoder = result["label_encoder"]
train_classes = pd.Series(encoder.inverse_transform(result["y_train"])).value_counts().sort_index()
test_classes = pd.Series(encoder.inverse_transform(result["y_test"])).value_counts().sort_index()
display(pd.DataFrame({"train": train_classes, "test": test_classes}).fillna(0).astype(int))
print("Urutan 13 kolom terverifikasi:", list(result["X_train"].columns) == FEATURES)
print("Nilai finite train/test:", bool(np.isfinite(result["X_train"].values).all()), bool(np.isfinite(result["X_test"].values).all()))

## 5. Visualisasi PCA data training (opsional untuk analisis awal)

PCA di-*fit* hanya pada training. Diagram di sini adalah **visualisasi EDA**, bukan hasil clustering K-Means.

In [ ]:
from sklearn.decomposition import PCA
pca = PCA(n_components=2, random_state=42)
projection = pca.fit_transform(result["X_train"])
names = encoder.inverse_transform(result["y_train"])
plt.figure(figsize=(9, 6))
sns.scatterplot(x=projection[:,0], y=projection[:,1], hue=names, s=20, alpha=.75)
plt.title(f"PCA training — explained variance {pca.explained_variance_ratio_.sum():.1%}")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

## 6. Verifikasi hasil ekspor dan kompatibilitas input web

Tes berikut membandingkan hasil `transform_input(9 fitur)` terhadap scaler yang sama, langsung pada satu baris data. Ini memastikan 4 fitur turunan dibuat dengan urutan yang benar dan scaling hanya sekali.

In [ ]:
artifacts = [
    OUT_DIR / "cwru_clean.csv", OUT_DIR / "train_scaled.csv", OUT_DIR / "test_scaled.csv",
    OUT_DIR / "split_manifest.csv", OUT_DIR / "preprocessing_summary.json",
    MODEL_DIR / "scaler.pkl", MODEL_DIR / "label_encoder.pkl",
    MODEL_DIR / "feature_list.json", MODEL_DIR / "feature_schema.json",
]
for path in artifacts:
    assert path.exists(), f"File belum dibuat: {path}"
    print(f"{path.relative_to(ROOT)} — {path.stat().st_size/1024:.1f} KB")

row9 = {name: float(result["df_clean"].iloc[0][name]) for name in BASE_FEATURES}
scaled_from_web = transform_input(row9, model_dir=MODEL_DIR)
scaled_direct = result["scaler"].transform(result["df_clean"].iloc[[0]][FEATURES])
np.testing.assert_allclose(scaled_from_web, scaled_direct, rtol=1e-10, atol=1e-10)
assert scaled_from_web.shape == (1, 13)
print("\nPASS — hasil web 9 fitur -> 13 scaled cocok dengan training")
print("Mapping kelas:", json.loads((MODEL_DIR/"feature_schema.json").read_text(encoding="utf-8"))["class_labels"])

## 7. Serah terima ke ML Engineer dan Backend

**ML Engineer 1 — Random Forest:** `train_scaled.csv`, `test_scaled.csv`, `label_encoder.pkl`. Ambil X dengan menghapus kolom `label`, y dari kolom `label`. Latih Random Forest langsung dengan **13 fitur scaled**.

**ML Engineer 2 — K-Means:** gunakan X dari CSV yang sama tanpa `label`; kolom label hanya untuk interpretasi/penilaian ARI/NMI, **jangan** dipakai untuk training clustering.

**Backend/Frontend:** input form web adalah 9 fitur dasar sesuai `models/feature_schema.json`. Panggil `preprocessing.transform_input(values, model_dir="models")` menjadi array `(1, 13)` sebelum `model.predict(...)`. Jangan menerapkan scaler lagi dalam model `.pkl`.

```python
# Contoh pembacaan untuk training (bukan training model):
train = pd.read_csv("data/processed/train_scaled.csv")
test = pd.read_csv("data/processed/test_scaled.csv")
X_train, y_train = train.drop(columns="label"), train["label"]
X_test, y_test = test.drop(columns="label"), test["label"]
```

Output model yang akan dikirim oleh masing-masing ML Engineer setelah training: `models/random_forest_pipeline.pkl` dan `models/kmeans_pipeline.pkl`. Meski bernama `pipeline`, isinya sebaiknya estimator terlatih pada data scaled untuk menghindari double scaling.

**Catatan:** file `feature_schema.json` yang baru perlu menggantikan skema 8 fitur lama pada template web; `backend/model_service.py` masih harus diubah agar benar-benar memanggil `transform_input` sebelum prediksi.

## 8. Kesimpulan

Isi ringkasan berdasarkan hasil **Run All** dan tangkapan grafik. Hindari mengutip nilai evaluasi model sebelum training dilakukan. Dokumentasikan konfigurasi CSV, jumlah kelas, pembagian data, fitur, dan keterbatasan split acak antar window.